# Modeling

## Data preparation

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from pathlib import Path

base_dir = Path.cwd().parent
raw_data = base_dir / "data" / "raw"
processed_data = base_dir / "data" / "processed"

df = pd.read_csv(raw_data / "predictive_maintenance.csv")

features = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

X = df[features]
y = df["Target"]


x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

ct = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Type']),         
        ('num', StandardScaler(), [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
    ])  
    ]
)

pipeline = Pipeline([
    ('preprocessing', ct),
    ('model', LogisticRegression(class_weight="balanced"))
])

## Model learning

In [2]:
pipeline.fit(x_train, y_train)

print("Model learning completed successfully!")

Model learning completed successfully!


## Evaluation
- the model “predicts failure” ~6x more often than it occurs
- the model detects most failures

In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import confusion_matrix

y_pred = pipeline.predict(x_test)

probabilities = pipeline.predict_proba(x_test)

print("Probabilities of failure: ")
print(probabilities[:, 1])

precision = precision_score(y_test, y_pred, average='binary')
recall = recall_score(y_test, y_pred, average='binary')
f1 = f1_score(y_test, y_pred, average='binary')

print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Probabilities of failure: 
[0.97135024 0.5379081  0.68019628 ... 0.14890765 0.59522328 0.01710193]
Precision: 0.14
Recall: 0.82
F1 Score: 0.24

Confusion Matrix:
[[1593  339]
 [  12   56]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.82      0.90      1932
           1       0.14      0.82      0.24        68

    accuracy                           0.82      2000
   macro avg       0.57      0.82      0.57      2000
weighted avg       0.96      0.82      0.88      2000



## Tresholds comparision
- Model can't separate classes well, it just shifts the decision boundary.
- Logistic Regression is too simple for this problem

In [8]:
thresholds = [0.5, 0.4, 0.3, 0.2]

for t in thresholds:
    y_pred = (probs >= t).astype(int)
    
    precision = precision_score(y_test, y_pred, average='binary')
    recall = recall_score(y_test, y_pred, average='binary')
    f1 = f1_score(y_test, y_pred, average='binary')
    
    print(t, precision, recall, f1)

0.5 0.14177215189873418 0.8235294117647058 0.24190064794816415
0.4 0.11509433962264151 0.8970588235294118 0.2040133779264214
0.3 0.09197080291970802 0.9264705882352942 0.16733067729083664
0.2 0.07246376811594203 0.9558823529411765 0.13471502590673576
